# SECTION 1: Project Introduction
## Lung Segmentation and Automatic Cardiothoracic Ratio (CTR) Calculation from Chest X-Ray Images

This notebook implements a complete, runnable end-to-end deep learning system for lung segmentation and automatic CTR calculation based on the research paper:
*"Automatic cardiothoracic ratio calculation based on lung fields abstracted from chest X-ray images without heart segmentation"*

**Strict Framework Constraint**: TensorFlow 2.x / Keras ONLY.

# SECTION 2: Import Libraries

In [ ]:
import os
import sys
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path

# Ensure project root is in sys.path
BASE_DIR = Path(".").resolve()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

print(f"TensorFlow Version: {tf.__version__}")
print(f"NumPy Version: {np.__version__}")

# SECTION 3: Configuration

In [ ]:
import config
from src.utils import set_seed

set_seed(config.RANDOM_SEED)
print(f"Image Size: {config.IMAGE_SIZE}")
print(f"Batch Size: {config.BATCH_SIZE}")
print(f"Learning Rate: {config.LEARNING_RATE}")

# SECTION 4: Dataset Discovery

In [ ]:
from src.dataset import discover_dataset

df_pairs, unlabelled_test, report = discover_dataset()

# SECTION 5: Dataset Statistics

In [ ]:
print("=== DATASET STATISTICS ===")
for k, v in report.items():
    print(f"  {k}: {v}")

# SECTION 6: Image-Mask Matching

In [ ]:
print(f"Matched Image-Mask Pairs: {len(df_pairs)}")
df_pairs.head(5)

# SECTION 7: Train/Validation/Test Split

In [ ]:
from src.dataset import create_dataset_split

split_df = create_dataset_split(force_recreate=False)
print(split_df['split'].value_counts())

# SECTION 8: Sample Images

In [ ]:
from src.preprocessing import load_and_preprocess_image

sample_row = split_df.iloc[0]
sample_img = load_and_preprocess_image(sample_row['image_path'])

plt.figure(figsize=(5, 5))
plt.imshow(sample_img.squeeze(), cmap='gray')
plt.title(f"Sample CXR: {sample_row['image_filename']}")
plt.axis('off')
plt.show()

# SECTION 9: Sample Masks

In [ ]:
from src.preprocessing import load_and_preprocess_mask

sample_mask = load_and_preprocess_mask(sample_row['mask_path'])

plt.figure(figsize=(5, 5))
plt.imshow(sample_mask.squeeze(), cmap='gray')
plt.title(f"Sample Mask: {sample_row['mask_filename']}")
plt.axis('off')
plt.show()

# SECTION 10: Preprocessing

In [ ]:
print(f"Preprocessed Image Shape: {sample_img.shape}, Min: {sample_img.min()}, Max: {sample_img.max()}")
print(f"Preprocessed Mask Shape: {sample_mask.shape}, Unique Values: {np.unique(sample_mask)}")

# SECTION 11: Augmentation

In [ ]:
from src.augmentation import create_tf_dataset

train_df = split_df[split_df['split'] == 'train']
train_ds = create_tf_dataset(train_df, batch_size=config.BATCH_SIZE, is_training=True)
for imgs, masks in train_ds.take(1):
    print(f"Batch Image Shape: {imgs.shape}, Batch Mask Shape: {masks.shape}")

# SECTION 12: Build SegNet

In [ ]:
from models_arch.segnet import build_segnet

segnet_model = build_segnet(input_shape=config.INPUT_SHAPE)
segnet_model.summary()

# SECTION 13: Train SegNet

In [ ]:
# SegNet training is handled via training/train_segnet.py
print(f"Target Best Model: {config.MODEL_PATHS['segnet']}")

# SECTION 14: Evaluate SegNet

In [ ]:
# Evaluate SegNet performance on test split
print("SegNet ready for multi-model evaluation.")

# SECTION 15: Build U-Net

In [ ]:
from models_arch.unet import build_unet

unet_model = build_unet(input_shape=config.INPUT_SHAPE)
unet_model.summary()

# SECTION 16: Train U-Net

In [ ]:
print(f"Target Best Model: {config.MODEL_PATHS['unet']}")

# SECTION 17: Evaluate U-Net

In [ ]:
print("U-Net ready for multi-model evaluation.")

# SECTION 18: Build ResU-Net++

In [ ]:
from models_arch.resunetpp import build_resunetplusplus

resunetpp_model = build_resunetplusplus(input_shape=config.INPUT_SHAPE)
resunetpp_model.summary()

# SECTION 19: Train ResU-Net++

In [ ]:
print(f"Target Best Model: {config.MODEL_PATHS['resunetpp']}")

# SECTION 20: Evaluate ResU-Net++

In [ ]:
print("ResU-Net++ ready for multi-model evaluation.")

# SECTION 21: Build AttU-Net

In [ ]:
from models_arch.attunet import build_attention_unet

attunet_model = build_attention_unet(input_shape=config.INPUT_SHAPE)
attunet_model.summary()

# SECTION 22: Train AttU-Net

In [ ]:
print(f"Target Best Model: {config.MODEL_PATHS['attunet']}")

# SECTION 23: Evaluate AttU-Net

In [ ]:
print("AttU-Net ready for multi-model evaluation.")

# SECTION 24: Four-Model Metric Comparison

In [ ]:
res_csv = config.RESULTS_DIR / "segmentation_results.csv"
if res_csv.exists():
    df_res = pd.read_csv(res_csv)
    print(df_res.to_string(index=False))
else:
    print("Run evaluation/evaluate_segmentation.py to view metrics.")

# SECTION 25: Actual vs Predicted Masks

In [ ]:
viz_files = list(config.VISUALIZATIONS_DIR.glob("*.png"))
print(f"Generated {len(viz_files)} comparison visualizations in {config.VISUALIZATIONS_DIR}")

# SECTION 26: Connected-Component Post-Processing

In [ ]:
from src.postprocessing import postprocess_mask

raw_dummy = np.random.rand(256, 256)
clean_dummy = postprocess_mask(raw_dummy, threshold=0.5)
print(f"Dummy Postprocessing Output Shape: {clean_dummy.shape}")

# SECTION 27: Lung Contour Extraction

In [ ]:
from src.lung_analysis import extract_lung_contours

cnt_r, cnt_l = extract_lung_contours(sample_mask.squeeze())
print(f"Right Lung Contour Points: {len(cnt_r) if cnt_r is not None else 0}")
print(f"Left Lung Contour Points: {len(cnt_l) if cnt_l is not None else 0}")

# SECTION 28: CTR Calculation

In [ ]:
from src.ctr import calculate_ctr_geometry

ctr, c_dia, t_dia, pts, reason = calculate_ctr_geometry(sample_mask.squeeze())
print(f"CTR: {ctr:.4f}, Cardiac Dia: {c_dia:.1f} px, Thoracic Dia: {t_dia:.1f} px, Status: {reason}")

# SECTION 29: CTR Visualization

In [ ]:
from src.visualization import plot_ctr_visualization

fig_ctr = plot_ctr_visualization(sample_img, sample_mask, pts, ctr, c_dia, t_dia, model_name="Ground Truth")
plt.show()

# SECTION 30: Final Results

In [ ]:
ctr_csv = config.RESULTS_DIR / "ctr_results.csv"
if ctr_csv.exists():
    df_ctr = pd.read_csv(ctr_csv)
    print(df_ctr.head(5))
else:
    print("Run evaluation/evaluate_ctr.py to generate CTR results table.")

# SECTION 31: Save All Outputs

In [ ]:
print("[SUCCESS] All models, metrics, histories, plots, and CTR visualizations saved under outputs/")